# Image Classification Model Example

This notebook builds a simple image classification model using Python and scikit-learn. The goal is to show a complete machine learning workflow for classifying images into categories.

To keep the notebook easy to run, it uses the built-in `digits` dataset from scikit-learn. Each image is an 8x8 grayscale image of a handwritten digit from 0 to 9.

The workflow includes:

1. Loading image data
2. Visualizing sample images
3. Preparing features and labels
4. Splitting the data into training and testing sets
5. Training an image classification model
6. Evaluating accuracy and errors
7. Testing predictions on sample images

This is a clean portfolio-style example because it shows the basic logic of image classification without requiring a large external dataset.

## 1. Import Libraries

We begin by importing the main libraries used in the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)

## 2. Load the Image Dataset

The digits dataset contains small grayscale images of handwritten numbers. Each image is stored as an 8 by 8 matrix of pixel values.

In [ ]:
digits = load_digits()

images = digits.images
X = digits.data
y = digits.target

print("Image array shape:", images.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Class labels:", digits.target_names)

## 3. Visualize Sample Images

Before modeling, it is useful to inspect a few images. Each image is low-resolution, but the digits are still usually recognizable.

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i], cmap="gray")
    plt.title(f"Label: {y[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Understand the Pixel Features

Each 8x8 image has 64 pixels. For modeling, the image is flattened into a row of 64 numerical features. Each feature represents the intensity of one pixel.

In [ ]:
pixel_df = pd.DataFrame(X, columns=[f"pixel_{i}" for i in range(X.shape[1])])
pixel_df["label"] = y

pixel_df.head()

## 5. Split the Data

The model is trained on one part of the data and tested on a separate part. This helps us evaluate whether the model can classify images it has not seen before.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

## 6. Train a Baseline Image Classification Model

We start with logistic regression. Even though the name says regression, logistic regression is commonly used for classification problems.

Because pixel values are numerical features, scaling can help the model train more effectively.

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(y_test, logistic_predictions)
print("Logistic Regression Accuracy:", round(logistic_accuracy, 4))

## 7. Train a Random Forest Image Classifier

A random forest classifier can capture nonlinear patterns in the pixel values. It is often a strong model for tabular-style machine learning problems.

In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42
)

forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

forest_accuracy = accuracy_score(y_test, forest_predictions)
print("Random Forest Accuracy:", round(forest_accuracy, 4))

## 8. Compare Model Performance

Now we compare the two models using accuracy. Accuracy measures the percentage of test images classified correctly.

In [ ]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [logistic_accuracy, forest_accuracy]
})

model_results

## 9. Classification Report

The classification report gives more detail by showing precision, recall, and F1-score for each digit.

In [ ]:
best_predictions = forest_predictions if forest_accuracy >= logistic_accuracy else logistic_predictions
best_model_name = "Random Forest" if forest_accuracy >= logistic_accuracy else "Logistic Regression"

print("Best model:", best_model_name)
print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

## 10. Confusion Matrix

The confusion matrix shows which digits were classified correctly and which digits were confused with each other.

In [ ]:
cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xticks(range(10))
plt.yticks(range(10))
plt.tight_layout()
plt.show()

## 11. View Correct and Incorrect Predictions

Looking at actual predictions helps us understand the model more intuitively.

In [ ]:
test_images = X_test.reshape(-1, 8, 8)

plt.figure(figsize=(10, 6))

for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(test_images[i], cmap="gray")
    plt.title(f"True: {y_test[i]} | Pred: {best_predictions[i]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 12. Find Misclassified Images

Misclassified examples are useful because they show where the model struggles.

In [ ]:
wrong_indexes = np.where(best_predictions != y_test)[0]

print("Number of misclassified images:", len(wrong_indexes))

if len(wrong_indexes) > 0:
    plt.figure(figsize=(10, 4))
    sample_wrong = wrong_indexes[:10]
    
    for plot_num, image_index in enumerate(sample_wrong):
        plt.subplot(2, 5, plot_num + 1)
        plt.imshow(test_images[image_index], cmap="gray")
        plt.title(f"True: {y_test[image_index]} | Pred: {best_predictions[image_index]}")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified images found in this test split.")

## 13. Predict One New Image

This section shows how the trained model can classify a single image from the test set.

In [ ]:
sample_index = 7
sample_image = X_test[sample_index].reshape(1, -1)
sample_prediction = forest_model.predict(sample_image)[0]

plt.figure(figsize=(3, 3))
plt.imshow(X_test[sample_index].reshape(8, 8), cmap="gray")
plt.title(f"Predicted Digit: {sample_prediction}")
plt.axis("off")
plt.show()

print("Actual label:", y_test[sample_index])
print("Predicted label:", sample_prediction)

## 14. Conclusion

This notebook demonstrated a complete image classification workflow. The images were converted into pixel-based numerical features, then two classification models were trained and evaluated.

The main idea is simple: an image can be represented as a collection of pixel values, and a machine learning model can learn patterns in those values that correspond to different classes.

For a more advanced version of this project, the next step would be to use a convolutional neural network, or CNN. CNNs are especially powerful for image classification because they can learn spatial patterns such as edges, curves, shapes, and textures directly from images.